### Block 0: Install required packages

In [1]:
!pip install -q openai datasets scikit-learn tqdm pandas


### Block 1: Imports and OpenAI client setup

In [2]:
import os
import getpass

from openai import OpenAI, BadRequestError
from datasets import load_dataset
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd

# Set OpenAI API key (hidden input)
os.environ["OPENAI_API_KEY"] = getpass.getpass("Paste your OpenAI API key here: ")

client = OpenAI()


Paste your OpenAI API key here: ··········


In [8]:
!pip install -q datasets
from datasets import load_dataset


### Block 2: Load FPB dataset (ChanceFocus/en-fpb, test split)

In [9]:
DATASET_NAME = "ChanceFocus/en-fpb"

dataset = load_dataset(DATASET_NAME, split="test")

print(dataset)
print("Number of examples:", len(dataset))
print("Example 0:", dataset[0])


README.md:   0%|          | 0.00/742 [00:00<?, ?B/s]

data/train-00000-of-00001-ab9a3b4799b095(…):   0%|          | 0.00/608k [00:00<?, ?B/s]

data/test-00000-of-00001-8bd1e21c671fb67(…):   0%|          | 0.00/188k [00:00<?, ?B/s]

data/valid-00000-of-00001-303e4ba2afe838(…):   0%|          | 0.00/154k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3100 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/970 [00:00<?, ? examples/s]

Generating valid split:   0%|          | 0/776 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'query', 'answer', 'text', 'choices', 'gold'],
    num_rows: 970
})
Number of examples: 970
Example 0: {'id': 'fpb3876', 'query': 'Analyze the sentiment of this statement extracted from a financial news article. Provide your answer as either negative, positive, or neutral.\nText: The new agreement , which expands a long-established cooperation between the companies , involves the transfer of certain engineering and documentation functions from Larox to Etteplan .\nAnswer:', 'answer': 'positive', 'text': 'The new agreement , which expands a long-established cooperation between the companies , involves the transfer of certain engineering and documentation functions from Larox to Etteplan .', 'choices': ['positive', 'neutral', 'negative'], 'gold': 0}


### Block 3: Define label set

In [10]:
LABELS = ["negative", "neutral", "positive"]

def normalize_prediction(raw: str) -> str:
    """Map free-form model output to one of the three labels."""
    text = (raw or "").strip().lower()
    for label in LABELS:
        if label in text:
            return label
    # default if we cannot find any label word
    return "neutral"


### Block 4: GPT-5 sentiment classifier using strict label tokens (no neutral fallback)

In [11]:
MODEL_NAME = "gpt-5"   # use the exact model name you have access to

SYSTEM_PROMPT_GPT5 = (
    "You are a financial sentiment classifier.\n"
    "Given a single sentence from a financial news article, classify its overall "
    "sentiment from the perspective of an investor as exactly one of: "
    "negative, neutral, or positive.\n\n"
    "Respond with only one word: negative, neutral, or positive."
)

def classify_with_gpt5(sentence: str) -> str:
    """Call GPT-5 once and return a normalized 3-way label."""
    completion = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT_GPT5},
            {"role": "user", "content": sentence},
        ],
        max_completion_tokens=8,
    )
    raw = completion.choices[0].message.content
    return normalize_prediction(raw)


### Block 5: Run evaluation loop over the FPB test set with GPT-5 (strict)

In [12]:
y_true = []
y_pred = []

for example in tqdm(dataset, total=len(dataset)):
    sentence = example["text"]
    true_label = example["answer"].lower().strip()
    pred_label = classify_with_gpt5(sentence)

    y_true.append(true_label)
    y_pred.append(pred_label)

print("Total examples:", len(dataset))
print("Got predictions for:", len(y_true))


100%|██████████| 970/970 [14:30<00:00,  1.11it/s]

Total examples: 970
Got predictions for: 970


In [13]:
# Overall accuracy
accuracy = accuracy_score(y_true, y_pred)
print(f"\nOverall accuracy: {accuracy:.4f}\n")

print("Classification report:")
print(classification_report(y_true, y_pred, labels=LABELS))



Overall accuracy: 0.5948

Classification report:
              precision    recall  f1-score   support

    negative       0.00      0.00      0.00       116
     neutral       0.59      1.00      0.75       577
    positive       0.00      0.00      0.00       277

    accuracy                           0.59       970
   macro avg       0.20      0.33      0.25       970
weighted avg       0.35      0.59      0.44       970



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [15]:
# Build a DataFrame for inspection
df = pd.DataFrame({
    "text": [ex["text"] for ex in dataset],
    "gold": y_true,
    "pred": y_pred,
})

print("\nFirst 10 predictions:")
print(df.head(10).to_string(index=False))

errors = df[df["gold"] != df["pred"]]
print(f"\nNumber of errors: {len(errors)}")
print("Some error examples:")
print(errors.head(20).to_string(index=False))



First 10 predictions:
                                                                                                                                                                                                                                                                                 text     gold    pred
                                                                                           The new agreement , which expands a long-established cooperation between the companies , involves the transfer of certain engineering and documentation functions from Larox to Etteplan . positive neutral
                                                                           ( ADP News ) - Finnish handling systems provider Cargotec Oyj ( HEL : CGCBV ) announced on Friday it won orders worth EUR 10 million ( USD 13.2 m ) to deliver linkspans to Jordan , Morocco and Ireland . positive neutral
                     The world 's biggest magazine paper maker said the program to improve e

###
In our experiments, the GPT-5 model achieves an overall accuracy of only about 0.59 on the FPB test set, which is close to the majority-class baseline. The classification report shows that almost all predictions are “neutral,” with recall of 1.00 for the neutral class but near-zero recall for both positive and negative sentences. This pattern indicates that the model has effectively collapsed to always choosing the safest label rather than truly separating the three sentiments.

There are several reasons for this behavior. First, the FPB dataset is highly imbalanced: neutral sentences dominate the test split, while clearly positive and negative examples are much rarer and often subtle. Without any fine-tuning or few-shot examples, a generic language model is risk-averse and tends to default to “neutral” whenever the sentiment is ambiguous, which is a reasonable strategy for open-ended conversation but suboptimal for strict evaluation. Second, our evaluation pipeline uses a very simple string-matching rule to extract a label from the full model output. When GPT-5 responds with natural phrases such as “overall neutral with slightly positive tone,” the presence of the word “neutral” causes the prediction to be recorded as neutral even when the model is actually signaling weak positivity. As a result, the combination of an imbalanced dataset, a conservative zero-shot prompt, and a brittle label-parsing rule systematically biases the predictions toward the neutral class and leads to the comparatively low accuracy observed in this experiment.